# RummaGEO Gene — chem-duplicate review

In [1]:
import gzip, re, uuid
from collections import defaultdict
from pathlib import Path
import pandas as pd
from maayanlab_bioinformatics.parse import gmt_read_iter

pd.set_option('display.max_rows', None)      # never truncate rows
pd.set_option('display.max_colwidth', 120)

MAIN = Path('/Users/johngardner_mountsinai/work/PerturbSeqr')
GENE_GMT = MAIN / 'db/data/gene_perturbation/rummageo-gene.gmt.gz'
CHEM_GMT = MAIN / 'db/data/drug_perturbation/rummageo-chem.gmt.gz'

def load_gmt(path):  # gzip adapter around maayanlab_bioinformatics' GMT parser
    with gzip.open(path, 'rt') as fh:
        return [(term, list(geneset)) for term, geneset in gmt_read_iter(fh)]

def gs_hash(genes):  # the project-specific hash the ingest uses (db/ingest.py)
    return str(uuid.uuid5(uuid.UUID(int=0), '\t'.join(sorted(set(genes)))))

def gse_of(term):
    m = re.search(r'GSE\d+', term)
    return m.group(0) if m else ''

def gene_pert_type(term):  # RummaGEO gene field 5 (parsed from condition title; often blank)
    f = term.rsplit(' ', 1)[0].split('::')
    return f[5] if len(f) > 5 else ''

gene = load_gmt(GENE_GMT)
chem = load_gmt(CHEM_GMT)
print(f'RummaGEO gene terms: {len(gene)}   chem terms: {len(chem)}')

RummaGEO gene terms: 7960   chem terms: 5345


## Duplicates: RummaGEO Gene ↔ RummaGEO Chem (byte-identical gene sets)

In [2]:
chem_by_hash = defaultdict(list)
for t, g in chem:
    chem_by_hash[gs_hash(g)].append(t)

dup_rows = []
for t, g in gene:
    twins = chem_by_hash.get(gs_hash(g))
    if not twins:
        continue
    direction = t.rsplit(' ', 1)[1]
    for chem_term in twins:
        dup_rows.append({
            'gse': gse_of(t) or gse_of(chem_term),
            'gene_pert_type': gene_pert_type(t),
            'gene_term': t,
            'chem_term': chem_term,
            'direction': direction,
            'n_genes': len(g),
        })

dup_df = pd.DataFrame(dup_rows).sort_values(['gse', 'gene_term']).reset_index(drop=True)
print(f'{len(dup_df)} duplicate terms ({dup_df["gene_term"].str.rsplit(" ", n=1).str[0].nunique()} signatures)')
with pd.option_context('display.max_rows', None):
    display(dup_df)

197 duplicate terms (104 signatures)


,gse,gene_pert_type,gene_term,chem_term,direction,n_genes
0,GSE123825,,glud1::::::::::::::mouse::GSE123825::0::3 down,glutamine::::::::::mouse::GSE123825::0::3 down,down,1352
1,GSE123825,,glud1::::::::::::::mouse::GSE123825::2::3 down,glutamine::::::::::mouse::GSE123825::2::3 down,down,699
2,GSE123825,,glud1::::::::::::::mouse::GSE123825::2::3 up,glutamine::::::::::mouse::GSE123825::2::3 up,up,1008
3,GSE126463,,arid2::::::::::::::human::GSE126463::0::2 down,pomalidomide::::::::::human::GSE126463::0::2 down,down,1300
4,GSE126463,,arid2::::::::::::::human::GSE126463::0::2 up,pomalidomide::::::::::human::GSE126463::0::2 up,up,772
5,GSE130396,,braf::::::::::::::human::GSE130396::0::2 up,neratinib::::::::::human::GSE130396::0::2 up,up,334
6,GSE130396,,braf::::::::::::::human::GSE130396::1::2 up,neratinib::::::::::human::GSE130396::1::2 up,up,10
7,GSE130396,,braf::::::::::::::human::GSE130396::7::6 down,neratinib::::::::::human::GSE130396::7::6 down,down,202
8,GSE130396,,braf::::::::::::::human::GSE130396::7::6 up,neratinib::::::::::human::GSE130396::7::6 up,up,159
9,GSE146982,,gpd1::::::::::::::human::GSE146982::0::1 down,metformin::::::::::human::GSE146982::0::1 down,down,773
